# pyKMC 

We gonna use a simple example of a Ni fcc system having a monovacancy to see how pyKMC works.  

In [1]:
#Some needed path :     
    #for pARTn and IRA
import sys, os
sys.path.insert(0,os.path.abspath('/root/programs/IterativeRotationsAssignments-master/interface'))
sys.path.insert(0,os.path.abspath('/root/programs/artn-plugin/interface'))
#%env PYTHONPATH=$PYTHONPATH:/root/programs/IterativeRotationsAssignements-master/interface:/root/programs/artn-plugin/interface
#For docker
%env OMPI_ALLOW_RUN_AS_ROOT=1
%env OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1

env: OMPI_ALLOW_RUN_AS_ROOT=1
env: OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1


In [2]:
sys.path

['/root/programs/artn-plugin/interface',
 '/root/programs/IterativeRotationsAssignments-master/interface',
 '/usr/lib/python310.zip',
 '/usr/lib/python3.10',
 '/usr/lib/python3.10/lib-dynload',
 '',
 '/root/pythonenv/pykmc/lib/python3.10/site-packages']

## System 

We will use [ASE](https://wiki.fysik.dtu.dk/ase/) to generate our simulation box : a Ni fcc cubic box of 2047 with one vacancy.

In [ ]:
from ase.build import bulk 
from ase.io import write 

#Parameters : 
output = './initial_config.xyz'
alat = 3.52 

atoms = bulk('Ni', crystalstructure='fcc', a=alat, cubic=True)
atoms = atoms.repeat((8,8,8))
atoms.pop(1166)
write(output, atoms)

We can view our system : 

In [ ]:
from ase.io import read 
from ase.visualize import view 

#Parameters : 
file_path = './initial_config.xyz'
atoms = read(file_path)


print("Number of atoms = {}".format(atoms.get_global_number_of_atoms()))
print("Cell is : \n {}".format(atoms.get_cell()))
view(atoms, viewer='ngl')


## Setup : 

pyKMC needs input parameters to execute all differents steps needed in a KMC simulation : 
- potential
- minimization 
- find atomic environments
- event search 
- point set registration 
- kmc 

pyKMC uses (for the moment) dictionaries to define parameters.

We first define a potential. We will use Lammps for all energies and forces calculcation, the dictionary associated to the potential is then composed of Lammps commands : 

In [3]:
pot_file_path = '../Ni_fcc_499at_monovacancy/Ni_v6_2.0_LKBeland2016.eam'
potential = {'pair_style' : 'eam/alloy', 
             'pair_coeff' : '* * {} Ni'.format(pot_file_path)}

For the minimization, we also need Lammps commands : 

In [4]:
minimization_parameters = {'min_style' : 'cg', 
                           'minimize' : '1.0e-6 1.0e-8 100 1000'}

For the atomic environment we need two parameters : 
- 'rcut' : radial cutoff defining the environement of an atom (use by 'graph') 
- 'rnei' : if the distance between two atoms is less than rnei, the two atoms are connected.
- 'radd_cna' : to add atoms at distance < radd_cna from non crystalline atoms (only with 'cna/graph' option) 

In [5]:
atomic_environment_parameters = {'rcut' : 5.0, 
                                 'rnei' : 3.0 ,
                                 'radd_cna' : 2.5}

For event searches we will use pARTn, we need : 
- 'nsearch' : nomber of event searches for each "ID" 
- 'path_artnso' : path to the pARTn Lammps plugin

In [6]:
search_parameters = {'nsearch' : 5, 
                     'path_artnso' : '/root/programs/artn-plugin/lib/libartn-lmp.so'}

## Parts of the KMC algorithm 

pyKMC uses a ```System``` class on which each part of the algorithm is build. So we first need to initialize the system and then we can apply different procedure to it, e.g. : 
```python 
system = System(init_config_file)
system.minimize(...)
...
system.kmc(...)
```


### Minimization : 

the ```system.minimize(...)```procedure perform a minimization and update the positions of the system.


In [ ]:
from pykmc.system import System

system = System('./initial_config.xyz')
system.minimize('lammps', minimization_parameters, potential)

when using Lammps, logs are save in lammps_minimization.log.

### Atomic environment : 

To associate generic event to an atom, pyKMC defines an "ID" for each atom in the system. Atoms having same "IDs" have same atomic environment. 
We will use the ```'cna/graph'``` option : 
1. Compute CNA with Lammps to find atoms that have a non crystalline environment.
2. For each atoms that have a non crystalline environment, and their nearest neighbors, we define their atomic environment using graphs.

In [ ]:
from pykmc.system import System 

system = System('./initial_config.xyz') 
system.minimize('lammps', minimization_parameters, potential)
system.find_environment('cna/graph', atomic_environment_parameters)

The results is a list of dictionaries, ```system.environment```, with the "IDs" and the list of atom's indexes with the corresponding "ID". 
We can check what we obtain by saving it to a yaml file : 

In [ ]:
import yaml 
with open('atomic_environment.yml', 'w') as outfile : 
    yaml.dump_all(system.environment, outfile, default_flow_style=False, sort_keys=False)

And we can also visualize it using ASE and by setting a different element type for each "ID" : 

In [ ]:
from ase import Atoms
from ase.visualize import view

#Create new Atoms : 
atoms = Atoms(positions = system.get_positions(), cell=system.get_cell(), pbc=True)

#Read atomic_environment file : 
with open('atomic_environment.yml', 'r') as file : 
    data = yaml.safe_load_all(file)
    l_id = list(data)

#Set element for each ID : 
z = [1]*atoms.get_global_number_of_atoms() 
for i, id in enumerate(l_id) : 
    at_idx = [int(e) for e in id["atom index"]]
    if id['ID'] != 'crystal' : 
        for at in at_idx : 
            z[at] = 30+i
atoms.set_atomic_numbers(z)

#view 
view(atoms, viewer='ngl')

### Event Search 

We use Lammps and pARTn. For each atomic environment "ID" that is not in the catalog we launch 'nsearch' pARTn searches and add the event to the ```system.catalog```catalog (and the backward reaction).

In [7]:
%%capture
from pykmc.system import System 

system = System('./initial_config.xyz')
system.minimize('lammps', minimization_parameters, potential)
system.find_environment('cna/graph', atomic_environment_parameters)
system.event_search('pARTn', search_parameters, potential)

Traceback (most recent call last):
  File "/root/pythonenv/pykmc/lib/python3.10/site-packages/executorlib/backend/interactive_serial.py", line 74, in <module>
    main(argument_lst=sys.argv)
  File "/root/pythonenv/pykmc/lib/python3.10/site-packages/executorlib/backend/interactive_serial.py", line 40, in main
    input_dict = interface_receive(socket=socket)
  File "/root/pythonenv/pykmc/lib/python3.10/site-packages/executorlib/standalone/interactive/communication.py", line 196, in interface_receive
    return cloudpickle.loads(socket.recv())
  File "/root/pyKMC/pykmc/__init__.py", line 4, in <module>
    from .event import *
  File "/root/pyKMC/pykmc/event.py", line 11, in <module>
    import pypARTn2
ModuleNotFoundError: No module named 'pypARTn2'


KeyboardInterrupt: 

The ```system.catalog```is a pandas DataFrame, that store needed informations of the event, we can display the energies and the rate constant of each event :  

In [ ]:
import pandas as pd 
catalog = system.catalog[['event_id', 'energy_barrier', 'k']]
catalog


We can also see the event :

In [ ]:
from ase import Atoms 
from ase.visualize import view

def display_event(idx_cat, system) : 
    event_traj = [] 
    c = ['initial_positions', 'saddle_positions', 'final_positions']
    central_atom = system.catalog.loc[idx_cat].at['central_atom'] 
    for e in c : 
        atoms = Atoms(positions=system.catalog.loc[idx_cat].at[e])
        z = atoms.get_global_number_of_atoms()*[30]
        z[central_atom] = 35
        atoms.set_atomic_numbers(z)
        event_traj.append(atoms)
    view(event_traj)

In [ ]:
display_event(0, system)

In [ ]:
from ase import Atoms 
from ase.visualize import view 
event_traj = [] 
c = ['initial_positions', 'saddle_positions', 'final_positions']
central_atom = system.catalog.loc[0].at['central_atom'] 
for e in c : 
    atoms = Atoms(positions=system.catalog.loc[0].at[e])
    z = atoms.get_global_number_of_atoms()*[30]
    z[central_atom] = 35
    atoms.set_atomic_numbers(z)
    event_traj.append(atoms)
view(event_traj[0])

<Popen: returncode: None args: ['/root/pythonenv/pykmc/bin/python', '-m', 'a...>

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,nomad-upload,nomad-get,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: couldn't connect to display ":3"
To get a full traceback, use: ase -T gui ...
